1. set up

In [5]:
import os
import pandas as pd
from qiime2 import Visualization
import matplotlib.pyplot as plt
import numpy as np

import qiime2 as q2

%matplotlib inline

# location of this week's data and all the results produced by this notebook 
# - this should be a path relative to your working directory
data_dir = '../data/processed_ignored'
results_dir = '../data/processed'

2. Download data from UNITE

In [5]:
!qiime rescript get-unite-data \
    --p-version 2025-02-19 \
    --p-taxon-group eukaryotes \
    --p-cluster-id dynamic \
    --p-no-singletons \
    --verbose \
    --output-dir uniteDB

Saved FeatureData[Taxonomy] to: uniteDB/taxonomy.qza
Saved FeatureData[Sequence] to: uniteDB/sequences.qza


3. Optional Curation

3.1 Remove sequences with unhelpful taxonomy: 

In [7]:
!qiime taxa filter-seqs \
    --p-exclude Fungi_sp,mycota_sp,mycetes_sp \
    --i-taxonomy uniteDB/taxonomy.qza \
    --i-sequences uniteDB/sequences.qza \
    --o-filtered-sequences uniteDB/sequences-filtered.qza

Saved FeatureData[Sequence] to: uniteDB/sequences-filtered.qza


3.2 Edit-taxonomy to make our classifier more efficient.

In [8]:
!qiime rescript edit-taxonomy \
    --i-taxonomy uniteDB/taxonomy.qza \
    --o-edited-taxonomy uniteDB/taxonomy-no-SH.qza \
    --p-search-strings ';sh__.*' \
    --p-replacement-strings '' \
    --p-use-regex

Saved FeatureData[Taxonomy] to: uniteDB/taxonomy-no-SH.qza


4. Train a naive-bayes classifier on UNITE （Not enough mem）

In [ ]:
#!qiime feature-classifier fit-classifier-naive-bayes \
    #--i-reference-reads uniteDB/sequences-filtered.qza \
    #--i-reference-taxonomy uniteDB/taxonomy-no-SH.qza \
    #--o-classifier uniteDB/classifier.qza

so used the trained one (https://github.com/colinbrislawn/unite-train/releases)

In [1]:
!wget -nc -O ../data/processed_ignored/unite_classifier_dynamic_euk.qza https://github.com/colinbrislawn/unite-train/releases/download/v10.0-2025-02-19-qiime2-2026.7/unite_ver2025-02-19_dynamic_eukaryotes-Q2-2026.7.qza
!wget -nc -O ../data/processed_ignored/unite_classifier_dynamic_fungi.qza https://github.com/colinbrislawn/unite-train/releases/download/v10.0-2025-02-19-qiime2-2026.7/unite_ver2025-02-19_dynamic_fungi-Q2-2026.7.qza
!wget -nc -O ../data/processed/eval_unite_ver2025-02-19.qzv https://github.com/colinbrislawn/unite-train/releases/download/v10.0-2025-02-19-qiime2-2026.7/eval_unite_ver2025-02-19.qzv

--2026-10-13 14:39:00--  https://github.com/colinbrislawn/unite-train/releases/download/v10.0-2025-02-19-qiime2-2026.7/unite_ver2025-02-19_dynamic_eukaryotes-Q2-2026.7.qza
Resolving github.com (github.com)... 20.250.119.67
Connecting to github.com (github.com)|20.250.119.67|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/448594227/abf98c9a-c8c9-4e56-be3d-0c8c8b37195d?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-10-13T13%3A25%3A29Z&rscd=attachment%3B+filename%3Dunite_ver2025-02-19_dynamic_eukaryotes-Q2-2026.7.qza&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-10-13T12%3A24%3A52Z&ske=2026-10-13T13%3A25%3A29Z&sks=b&skv=2018-11-09&sig=IVRTouGv3KpQlMHyqnhwgaVGOfapqcWHHuMEe87wlLM%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwi

5. Evaluate the classifier

In [ ]:
#!qiime rescript evaluate-fit-classifier \
    #--i-sequences uniteDB/sequences-filtered.qza   \
    #--i-taxonomy uniteDB/taxonomy-no-SH.qza \
    #--p-n-jobs 2 \
    #--o-classifier uniteDB/classifier.qza \
    #--o-evaluation uniteDB/classifier-evaluation.qzv \
    #--o-observed-taxonomy uniteDB/predicted-taxonomy.qza

#!qiime rescript evaluate-taxonomy \
  #--i-taxonomies uniteDB/taxonomy-no-SH.qza uniteDB/predicted-taxonomy.qza \
  #--p-labels ref-taxonomy predicted-taxonomy \
  #--o-taxonomy-stats uniteDB/both-taxonomy-evaluation.qzv

6. Taxonomy assignment

In [7]:
!qiime feature-classifier classify-sklearn \
  --i-classifier $data_dir/unite_classifier_dynamic_fungi.qza \
  --i-reads $data_dir/dada2_rep_set.qza \
  --p-n-jobs 1 \
  --p-reads-per-batch 500 \
  --o-classification $data_dir/taxonomy_unite_fungi.qza

Saved FeatureData[Taxonomy] to: ../data/processed_ignored/taxonomy_unite_fungi.qza


7. Taxonomy visualization

In [10]:
!qiime metadata tabulate \
  --m-input-file ../data/processed_ignored/taxonomy_unite_fungi.qza \
  --o-visualization ../data/processed/taxonomy_unite_fungi.qzv

Saved Visualization to: ../data/processed/taxonomy_unite_fungi.qzv


In [11]:
!qiime taxa filter-table \
  --i-table ../data/processed_ignored/dada2_table.qza \
  --i-taxonomy ../data/processed_ignored/taxonomy_unite_fungi.qza \
  --p-include p__ \
  --p-exclude Unassigned \
  --o-filtered-table ../data/processed_ignored/table_fungi.qza

!qiime taxa filter-seqs \
  --i-sequences ../data/processed_ignored/dada2_rep_set.qza \
  --i-taxonomy ../data/processed_ignored/taxonomy_unite_fungi.qza \
  --p-include p__ \
  --p-exclude Unassigned \
  --o-filtered-sequences ../data/processed_ignored/rep_set_fungi.qza

Saved FeatureTable[Frequency] to: ../data/processed_ignored/table_fungi.qza
Saved FeatureData[Sequence] to: ../data/processed_ignored/rep_set_fungi.qza


In [14]:
   import pandas as pd

   md = pd.read_csv('../data/raw/fungut_metadata.tsv', sep='\t')
   print(md.columns.tolist()[:5])

   md = md.rename(columns={'uniqueID': 'sample-id'})
   md.to_csv('../data/processed_ignored/fungut_metadata_clean.tsv', sep='\t', index=False)
   print('saved')

['ID', 'country_sample', 'state_sample', 'latitude_sample', 'longitude_sample']
saved


In [15]:
!qiime feature-table summarize \
  --i-table ../data/processed_ignored/table_fungi.qza \
  --m-metadata-file ../data/processed_ignored/fungut_metadata_clean.tsv \
  --o-feature-frequencies ../data/processed_ignored/table_fungi_feature_frequencies.qza \
  --o-sample-frequencies ../data/processed_ignored/table_fungi_sample_frequencies.qza \
  --o-summary ../data/processed/table_fungi.qzv

Saved ImmutableMetadata to: ../data/processed_ignored/table_fungi_feature_frequencies.qza
Saved ImmutableMetadata to: ../data/processed_ignored/table_fungi_sample_frequencies.qza
Saved Visualization to: ../data/processed/table_fungi.qzv


In [16]:
!qiime taxa barplot \
  --i-table ../data/processed_ignored/table_fungi.qza \
  --i-taxonomy ../data/processed_ignored/taxonomy_unite_fungi.qza \
  --m-metadata-file ../data/processed_ignored/fungut_metadata_clean.tsv \
  --o-visualization ../data/processed/taxa_barplot_fungi.qzv

Saved Visualization to: ../data/processed/taxa_barplot_fungi.qzv


In [17]:
from qiime2 import Visualization
Visualization.load('../data/processed/taxa_barplot_fungi.qzv')

<visualization: Visualization uuid: 53593c49-e92e-4ec2-8807-cb060b899b77>